# nb_00_bootstrap - reference tables and history files, straight from GitHub

**Use this only if you skipped Part 2 or Lab 1.** It gives `lh_brightcart` everything the later notebooks need:

- `ref_carriers`, `ref_warehouses`, `ref_regions`, `ref_calendar` as Delta tables (normally these are OneLake shortcuts to `lh_reference`)
- the five history CSVs in `Files/bronze/history/` (normally landed by the `pl_ingest_history` pipeline)

Attach `lh_brightcart` as the default lakehouse before you run it. ⟲ verify at recording: the lakehouse explorer shows it as the pinned (default) lakehouse.

**Parameters.** `base_url` is the folder that holds `manifest.json` (raw GitHub URL of the repo's `data/` folder). `overwrite_refs` protects existing shortcuts.

In [ ]:
base_url = "https://raw.githubusercontent.com/lostspace003/fabric-course/main/data"
overwrite_refs = False   # True = replace existing ref_* tables. Leave False when they are OneLake shortcuts to lh_reference.

**Read the manifest.** It lists every file with its URL and row count, so this notebook never hard-codes file names.

In [ ]:
import json, urllib.request
import pandas as pd

with urllib.request.urlopen(f"{base_url}/manifest.json", timeout=60) as r:
    manifest = json.load(r)
files = {f["path"]: f for f in manifest["files"]}
print(f'{len(files)} files in the manifest, seed {manifest["seed"]}')

**The calendar rule.** `df_calendar` (Dataflow Gen2, Part 2) adds four columns to `calendar_raw.csv`; this is the same rule in pandas so both paths give identical `ref_calendar` tables.

In [ ]:
def enrich_calendar(cal: pd.DataFrame) -> pd.DataFrame:
    """Adds dow, is_weekend, is_holiday, is_peak - the same rule as df_calendar (Dataflow Gen2) in Part 2."""
    d = pd.to_datetime(cal["date"]); md = d.dt.strftime("%m-%d")
    cal = cal.copy()
    cal["dow"] = d.dt.day_of_week                                   # Monday = 0
    cal["is_weekend"] = (cal.dow >= 5).astype(int)
    cal["is_holiday"] = cal.holiday_name.notna().astype(int)
    cal["is_peak"] = (((md >= "11-15") & (md <= "12-31")) | cal.promo_event.notna()).astype(int)   # 15 Nov-31 Dec or a promo week
    return cal

**Load the four reference tables.** Each CSV is read with pandas, typed, converted to Spark and saved as a Delta table. Existing tables are skipped unless `overwrite_refs` is True.

In [ ]:
def save_ref(table, pdf):
    if spark.catalog.tableExists(table) and not overwrite_refs:
        print(f"skip  {table}: already exists (shortcut or table). Set overwrite_refs=True to replace it.")
        return
    spark.createDataFrame(pdf).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"wrote {table}: {len(pdf)} rows")

carriers = pd.read_csv(files["reference/carriers.csv"]["url"])
warehouses = pd.read_csv(files["reference/warehouses.csv"]["url"])
regions = pd.read_csv(files["reference/regions.csv"]["url"])
calendar = enrich_calendar(pd.read_csv(files["reference/calendar_raw.csv"]["url"]))
calendar["date"] = pd.to_datetime(calendar["date"]).dt.date          # a real DATE column in Delta
calendar["holiday_name"] = calendar["holiday_name"].astype("string")  # nullable text, not a float column of NaN
calendar["promo_event"] = calendar["promo_event"].astype("string")

save_ref("ref_carriers", carriers)
save_ref("ref_warehouses", warehouses)
save_ref("ref_regions", regions)
save_ref("ref_calendar", calendar)

**Download the five history files into `Files/bronze/history/`.** Same place the pipeline writes to, so `nb_01` works either way.

In [ ]:
notebookutils.fs.mkdirs("Files/bronze/history")
for name in manifest["history_files"]:
    url = files[f"history/{name}"]["url"]
    with urllib.request.urlopen(url, timeout=120) as r:
        text = r.read().decode("utf-8")
    notebookutils.fs.put(f"Files/bronze/history/{name}", text, True)   # True = overwrite
    print(f"saved Files/bronze/history/{name}  ({len(text)/1e6:.1f} MB)")

**Check cell (Lab 1 checkpoint).** Row counts of the landed files must match the manifest. Duplicates are expected in `orders.csv` (about 0.5%); Lab 2 removes them.

In [ ]:
ok = True
for name in manifest["history_files"]:
    n = spark.read.option("header", True).csv(f"Files/bronze/history/{name}").count()
    want = files[f"history/{name}"]["rows"]
    flag = "OK " if n == want else "BAD"
    ok &= n == want
    print(f"{flag} {name:<20} {n:>7} rows (manifest {want})")
print("ALL FILES MATCH THE MANIFEST" if ok else "MISMATCH: re-run the pipeline or this notebook")